In [ ]:
import pandas as pd



In [ ]:
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
geolocation = pd.read_csv("../data/raw/olist_geolocation_dataset.csv")
order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")
reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
products = pd.read_csv("../data/raw/olist_products_dataset.csv")
sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")
category_translation = pd.read_csv("../data/raw/product_category_name_translation.csv")

#Dictionary so the prevent repition
datasets = {
    "customers": customers,
    "geolocation": geolocation,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation
}

In [ ]:
#size of every dataset
for name, df in datasets.items():
    print(f"{name}: {df.shape}")


In [ ]:
customers.head()

In [ ]:
customers.info()

In [ ]:
customers.isnull().sum()

In [ ]:
customers.duplicated().sum()

In [ ]:
customers["customer_id"].duplicated().sum() #check if customer_id is PRIMARY KEY

### Customers Table

- Each row represents a customer associated with an order.
- `customer_id` is unique for every row and can be treated as the primary key.
- `customer_unique_id` identifies the same customer across multiple orders.
- Customer location information is provided using ZIP prefix, city, and state.

In [ ]:
orders.head()

In [ ]:
orders.info()

In [ ]:
orders.isnull().sum()


In [ ]:
orders["order_id"].duplicated().sum() #checking order_id is PRIMARY KEY

In [ ]:
orders["order_status"].value_counts()

In [ ]:
orders.groupby("order_status")[
    [
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date"
    ]
].apply(lambda x: x.isnull().sum())

In [ ]:
orders[orders["order_delivered_customer_date"].isnull()][
    ["order_id", "order_status", "order_delivered_customer_date"]
].head(20)

In [ ]:
orders["order_status"].value_counts()

In [ ]:
missing_by_status = (
    orders.groupby("order_status")[
        [
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date"
        ]
    ]
    .apply(lambda x: x.isnull().mean() * 100)
    .round(2)
)

missing_by_status

### Missing-value observations

Missing timestamps are not necessarily data-quality errors.

- Shipped orders are expected to lack a customer delivery date.
- Processing/invoiced orders may legitimately lack carrier and delivery dates.
- Canceled orders may have progressed through approval or shipping before cancellation.
- A small number of orders marked as delivered lack expected lifecycle timestamps, indicating incomplete or inconsistent records that require further investigation.

In [ ]:
orders[
    (orders["order_status"] == "canceled") &
    (orders["order_delivered_customer_date"].notna())
][
    [
        "order_id",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date"
    ]
]

### Cancelled orders with recorded deliveries

Six orders are marked as `canceled` despite having a recorded customer
delivery timestamp.

The timestamps indicate that these orders progressed through purchase,
approval, carrier handoff and customer delivery.

Because the dataset does not contain a cancellation timestamp or additional
information explaining these cases, their true outcome cannot be determined
with certainty. These records will therefore be retained and flagged as
potential lifecycle inconsistencies rather than manually corrected.

In [ ]:
orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].isna())
][
    [
        "order_id",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
]

### Delivered orders with missing delivery timestamps

Eight orders are marked as `delivered` but do not contain an
`order_delivered_customer_date`.

Most of these records contain purchase, approval and carrier timestamps,
suggesting that the order progressed through the delivery process but the
final delivery timestamp was not recorded.

One delivered order is also missing its carrier handoff timestamp.

These records will be retained but excluded from analyses that require an
actual delivery timestamp, such as delivery duration and delivery-delay
calculations.

In [ ]:
orders[
    (orders["order_status"] == "delivered") &
    (orders["order_approved_at"].isna())
][
    [
        "order_id",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
]

### Delivered orders with missing approval timestamps

Fourteen orders are marked as delivered but have no recorded
`order_approved_at` timestamp.

All fourteen contain carrier and customer delivery timestamps, indicating
that the orders progressed through fulfillment and were successfully delivered.

The missing approval timestamps are therefore treated as incomplete lifecycle
records rather than evidence that the orders were not approved.

In [ ]:
order_items.head()

In [ ]:
order_items.info()

In [ ]:
order_items.isnull().sum()

In [ ]:
order_items.duplicated().sum()

In [ ]:
print("Rows:", len(order_items))
print("Unique orders:", order_items["order_id"].nunique())

In [ ]:
order_items.duplicated(
    subset=["order_id", "order_item_id"]
).sum()

In [ ]:
order_items.groupby("order_id")["order_item_id"].max().value_counts().sort_index()

In [ ]:
order_items["order_item_id"].max()

In [ ]:
order_items.isnull().sum()

In [ ]:
order_items[["price", "freight_value"]].describe()

In [ ]:
print("Zero prices:", (order_items["price"] == 0).sum())
print("Negative prices:", (order_items["price"] < 0).sum())

print("Zero freight:", (order_items["freight_value"] == 0).sum())
print("Negative freight:", (order_items["freight_value"] < 0).sum())

In [ ]:
print("99th percentile price:",
      order_items["price"].quantile(0.99))

print("99th percentile freight:",
      order_items["freight_value"].quantile(0.99))

In [ ]:
order_items.sort_values( "price",ascending=False).head(10)

In [ ]:
order_items.sort_values("freight_value",ascending=False).head(10)

### Order Items Data Quality Observations

- No missing values were found in the `order_items` table.
- No duplicate `(order_id, order_item_id)` combinations were found.
- Product prices are all positive.
- Freight values contain no negative values.
- 383 order items have zero freight cost, which may represent free-shipping transactions.
- Several high-value price and freight observations exist above the 99th percentile.
  These are retained because there is currently no evidence that they are erroneous.

In [ ]:
expensive_items = (
    order_items
    .sort_values("price", ascending=False)
    .head(10)
    .merge(
        products,
        on="product_id",
        how="left"
    )
)

expensive_items[
    [
        "product_id",
        "product_category_name",
        "price",
        "freight_value"
    ]
]

expensive_items = expensive_items.merge(
    category_translation,
    on="product_category_name",
    how="left"
)

expensive_items[
    [
        "product_category_name_english",
        "price",
        "freight_value"
    ]
]

In [ ]:
payments.head()

In [ ]:
payments.info()

In [ ]:
payments.isnull().sum()

In [ ]:
print("Rows:", len(payments))
print("Unique orders:", payments["order_id"].nunique())

In [ ]:
payments.duplicated(
    subset=["order_id", "payment_sequential"]
).sum()

In [ ]:
payments["payment_type"].value_counts()

In [ ]:
payments[["payment_installments", "payment_value"]].describe()

In [ ]:
print("Zero payment values:", (payments["payment_value"] == 0).sum())
print("Negative payment values:", (payments["payment_value"] < 0).sum())

print("Zero installments:", (payments["payment_installments"] == 0).sum())
print("Negative installments:", (payments["payment_installments"] < 0).sum())

In [ ]:
payments[payments["payment_value"] == 0]

In [ ]:
zero_payment_orders = payments.loc[
    payments["payment_value"] == 0,
    "order_id"
]

payments[
    payments["order_id"].isin(zero_payment_orders)
].sort_values(["order_id", "payment_sequential"])

In [ ]:
payments[
    payments["payment_installments"] == 0
]

In [ ]:
payments[
    payments["payment_type"] == "not_defined"
]

In [ ]:
zero_installment_orders = payments.loc[
    payments["payment_installments"] == 0,
    "order_id"
]

payments[
    payments["order_id"].isin(zero_installment_orders)
].sort_values(["order_id", "payment_sequential"])

### Payment Data Quality Observations

- No missing values or duplicate `(order_id, payment_sequential)` combinations were found.
- Multiple payment records per order are valid and may represent split payments.
- Nine payment records have a value of zero.
- Six of the zero-value records use vouchers, while three have `payment_type = not_defined`.
- Two credit-card records have `payment_installments = 0`. Since a single-payment credit-card transaction would normally use one installment, these are treated as anomalous installment values rather than evidence of a zero-installment payment.

### Zero-installment payment records

Two delivered orders contain credit-card payment records with
`payment_installments = 0`.

Both records also have `payment_sequential = 2`, despite there being no
corresponding sequence 1 payment record in the dataset.

The recorded payment values exactly match the associated order item and
freight totals, suggesting that the monetary values are valid while the
payment metadata is incomplete or inconsistent.

These records will therefore be retained and flagged rather than corrected
by assumption.

In [ ]:
orders_without_payment = orders[
    ~orders["order_id"].isin(payments["order_id"])
]

orders_without_payment

In [ ]:
order_items[
    order_items["order_id"] == "bfbd0f9bdef84302105ad712db648a6c"
]

### Order without a payment record

One order exists in the orders and order-items tables but has no corresponding
record in the payments table.

The order is marked as delivered and contains three order items. Based on
item prices and freight charges, the associated order value is 143.46.

Because there is no evidence explaining the missing payment record, the order
will be retained but excluded from analyses requiring recorded payment
information.

In [ ]:
reviews.head()
reviews.info()

In [ ]:
reviews.isnull().sum()

In [ ]:
reviews.duplicated().sum()

In [ ]:
print("Rows:", len(reviews))
print("Unique review IDs:", reviews["review_id"].nunique())
print("Unique orders:", reviews["order_id"].nunique())

In [ ]:
reviews["review_score"].value_counts().sort_index()

In [ ]:
reviews["review_id"].duplicated().sum()

In [ ]:
reviews["order_id"].duplicated().sum()

In [ ]:
reviews.duplicated(
    subset=["review_id", "order_id"]
).sum()

In [ ]:
duplicate_review_ids = reviews[
    reviews["review_id"].duplicated(keep=False)
].sort_values("review_id")

duplicate_review_ids.head(20)

In [ ]:
multiple_reviews_per_order = reviews[
    reviews["order_id"].duplicated(keep=False)
].sort_values(["order_id", "review_creation_date"])

multiple_reviews_per_order.head(20)

In [ ]:
reviews.groupby("order_id")["review_score"].nunique().value_counts()

### Review Identifier Observations

- No exact duplicate rows were found.
- `review_id` is not unique across the dataset.
- `order_id` is also not unique because some orders have multiple review records.
- The combination of `review_id` and `order_id` is unique.
- Repeated review IDs contain identical review information but are associated with
  different order IDs, indicating a dataset relationship inconsistency or source-system
  characteristic rather than simple duplicate rows.
- Some orders contain multiple distinct reviews and therefore require further
  investigation before creating order-level satisfaction metrics.

In [ ]:
products.head()

In [ ]:
products.info()

In [ ]:
products.isnull().sum()

In [ ]:
products.duplicated().sum()

In [ ]:
print("Rows:", len(products))
print("Unique product IDs:", products["product_id"].nunique())
print("Duplicate product IDs:", products["product_id"].duplicated().sum())

In [ ]:
products[
    [
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm",
        "product_photos_qty"
    ]
].describe()

In [ ]:
print("Zero weights:", (products["product_weight_g"] == 0).sum())
print("Negative weights:", (products["product_weight_g"] < 0).sum())

print("Zero lengths:", (products["product_length_cm"] == 0).sum())
print("Zero heights:", (products["product_height_cm"] == 0).sum())
print("Zero widths:", (products["product_width_cm"] == 0).sum())

In [ ]:
missing_metadata = products[
    products["product_category_name"].isna()
]

missing_metadata[
    [
        "product_category_name",
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty"
    ]
].isnull().sum()


In [ ]:
missing_metadata.head(10)

In [ ]:


products[
    products[
        [
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        ]
    ].isnull().any(axis=1)
]

In [ ]:
products[
    products["product_weight_g"] == 0
]

In [ ]:
zero_weight_products = (
    products[
        products["product_weight_g"] == 0
    ]
    .merge(
        category_translation,
        on="product_category_name",
        how="left"
    )
)

zero_weight_products[
    [
        "product_id",
        "product_category_name_english",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
]

### Product Data Quality Observations

- `product_id` is unique across all 32,951 products.
- No exact duplicate product records were found.
- 610 products are missing category and descriptive metadata
  (`product_category_name`, name length, description length and photo count).
  Most still contain valid physical measurements and will therefore be retained.
- Two products are missing all weight and dimension measurements.
- Four products have a recorded weight of 0 g despite having non-zero physical
  dimensions. These values are considered invalid physical measurements rather
  than genuine zero-weight products.
- No negative weights or dimensions were found.
- Large maximum weights and dimensions are retained because they are physically
  plausible and there is no evidence that they are erroneous.

In [ ]:
sellers.head()
sellers.info()
sellers.isnull().sum()
sellers.duplicated().sum()

In [ ]:
print("Rows:", len(sellers))
print("Unique seller IDs:", sellers["seller_id"].nunique())
print("Duplicate seller IDs:", sellers["seller_id"].duplicated().sum())

In [ ]:
print("Unique states:", sellers["seller_state"].nunique())
print("Unique cities:", sellers["seller_city"].nunique())

sellers["seller_state"].value_counts()

### Seller Data Quality Observations

- The sellers table contains 3,095 seller records.
- `seller_id` is unique and can be treated as the primary key.
- No missing values or duplicate seller records were found.
- Sellers are distributed across 23 states and 611 cities.
- Seller activity is geographically concentrated, particularly in São Paulo (`SP`).

In [ ]:
geolocation.head()

In [ ]:
geolocation.info()

In [ ]:
geolocation.isnull().sum()

In [ ]:
geolocation.duplicated().sum()

In [ ]:
print("Rows:", len(geolocation))

print(
    "Unique ZIP prefixes:",
    geolocation["geolocation_zip_code_prefix"].nunique()
)

print(
    "Unique cities:",
    geolocation["geolocation_city"].nunique()
)

print(
    "Unique states:",
    geolocation["geolocation_state"].nunique()
)

In [ ]:
geolocation[
    "geolocation_zip_code_prefix"
].value_counts().head(10)

In [ ]:
geolocation[
    ["geolocation_lat", "geolocation_lng"]
].describe()

In [ ]:
print(
    "Invalid latitude:",
    ((geolocation["geolocation_lat"] < -90) |
     (geolocation["geolocation_lat"] > 90)).sum()
)

print(
    "Invalid longitude:",
    ((geolocation["geolocation_lng"] < -180) |
     (geolocation["geolocation_lng"] > 180)).sum()
)

In [ ]:
geo_outliers = geolocation[
    (geolocation["geolocation_lat"] < -35) |
    (geolocation["geolocation_lat"] > 6) |
    (geolocation["geolocation_lng"] < -75) |
    (geolocation["geolocation_lng"] > -32)
]

print("Potential geographic outliers:", len(geo_outliers))

geo_outliers.head(20)

In [ ]:
geo_outliers[
    [
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng",
        "geolocation_city",
        "geolocation_state"
    ]
].sort_values("geolocation_lat")

In [ ]:
category_translation.head()

In [ ]:
category_translation.info()

In [ ]:
category_translation.isnull().sum()

In [ ]:
category_translation.duplicated().sum()

In [ ]:
print("Rows:", len(category_translation))
print(
    "Unique Portuguese categories:",
    category_translation["product_category_name"].nunique()
)
print(
    "Unique English categories:",
    category_translation["product_category_name_english"].nunique()
)

In [ ]:
product_categories = set(
    products["product_category_name"].dropna().unique()
)

translated_categories = set(
    category_translation["product_category_name"].unique()
)

missing_translations = product_categories - translated_categories

print("Missing translations:", missing_translations)

In [ ]:
print(
    "Number of missing translations:",
    len(missing_translations)
)

In [ ]:
missing_customers = orders[
    ~orders["customer_id"].isin(customers["customer_id"])
]

print("Orders with missing customer:", len(missing_customers))

In [ ]:
missing_orders_items = order_items[
    ~order_items["order_id"].isin(orders["order_id"])
]

print("Order items with missing order:", len(missing_orders_items))

In [ ]:
missing_products = order_items[
    ~order_items["product_id"].isin(products["product_id"])
]

print("Order items with missing product:", len(missing_products))

In [ ]:
missing_sellers = order_items[
    ~order_items["seller_id"].isin(sellers["seller_id"])
]

print("Order items with missing seller:", len(missing_sellers))

In [ ]:
missing_payment_orders = payments[
    ~payments["order_id"].isin(orders["order_id"])
]

print("Payments with missing order:", len(missing_payment_orders))

In [ ]:
geo_zips = set(
    geolocation["geolocation_zip_code_prefix"].unique()
)

customer_zips = set(
    customers["customer_zip_code_prefix"].unique()
)

seller_zips = set(
    sellers["seller_zip_code_prefix"].unique()
)

print(
    "Customer ZIP prefixes missing from geolocation:",
    len(customer_zips - geo_zips)
)

print(
    "Seller ZIP prefixes missing from geolocation:",
    len(seller_zips - geo_zips)
)

In [ ]:
missing_customer_zips = customer_zips - geo_zips
missing_seller_zips = seller_zips - geo_zips

affected_customers = customers[
    customers["customer_zip_code_prefix"].isin(missing_customer_zips)
]

affected_sellers = sellers[
    sellers["seller_zip_code_prefix"].isin(missing_seller_zips)
]

print("Affected customer records:", len(affected_customers))
print(
    "Affected customer %:",
    round(len(affected_customers) / len(customers) * 100, 2)
)

print("Affected seller records:", len(affected_sellers))
print(
    "Affected seller %:",
    round(len(affected_sellers) / len(sellers) * 100, 2)
)

In [ ]:
affected_orders = orders[
    orders["customer_id"].isin(affected_customers["customer_id"])
]

print("Orders without customer geolocation:", len(affected_orders))
print(
    "Percentage of orders:",
    round(len(affected_orders) / len(orders) * 100, 2)
)

In [ ]:
affected_seller_items = order_items[
    order_items["seller_id"].isin(affected_sellers["seller_id"])
]

print(
    "Order items from sellers without geolocation:",
    len(affected_seller_items)
)